In [ ]:
from google.colab import files
from bs4 import BeautifulSoup
import requests
import pandas as pd
import json
import time

In [ ]:
base_url = "https://www.panamericana.com.co"
url_cat = (
    "https://www.panamericana.com.co/13331"
    "?map=productClusterIds&order=OrderByTopSaleDESC"
)

pag_descargar = 2

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

In [ ]:
libros_lista = []

for pagina in range(1, pag_descargar + 1):
    url_actual = f"{url_cat}&page={pagina}"
    print(f"Leyendo página {pagina}...")

    response = requests.get(url_actual, headers=headers, timeout=15)

    if response.status_code != 200:
        print(f"Error {response.status_code} en página {pagina}, se omite.")
        continue

    soup = BeautifulSoup(response.content, "html.parser")
    scripts_json_ld = soup.find_all("script", type="application/ld+json")

    for script in scripts_json_ld:
        try:
            contenido_json = script.get_text(strip=True)
            if not contenido_json:
                continue

            datos = json.loads(contenido_json)

            if datos.get("@type") == "ItemList" and "itemListElement" in datos:
                for item_wrapper in datos["itemListElement"]:
                    producto = item_wrapper.get("item", {})
                    if not producto:
                        continue

                    codigo = producto.get("sku", producto.get("id", "No disponible"))
                    nombre = producto.get("name", "No disponible")

                    url_libro = producto.get("@id") or producto.get(
                        "url", "No disponible"
                    )
                    if url_libro != "No disponible" and not url_libro.startswith("http"):
                        url_libro = base_url + url_libro

                    ofertas = producto.get("offers", {})
                    precio = "No disponible"

                    if isinstance(ofertas, dict):
                        precio_num = (
                            ofertas.get("highPrice")
                            or ofertas.get("lowPrice")
                            or ofertas.get("price")
                        )
                        if precio_num:
                            precio = f"${float(precio_num):,.2f}"

                    libros_lista.append(
                        {
                            "Código": codigo,
                            "Nombre": nombre,
                            "Valor": precio,
                            "URL": url_libro,
                        }
                    )

        except (json.JSONDecodeError, TypeError, ValueError) as error:
            print(f"No se pudo leer un bloque JSON-LD: {error}")
            continue

    time.sleep(1)

Leyendo página 1...
Leyendo página 2...


In [ ]:
df_libros = pd.DataFrame(libros_lista)

if not df_libros.empty:
    df_libros = df_libros.drop_duplicates(subset=["Nombre"])

print(f"\nExtracción finalizada. Total de libros: {len(df_libros)}")
display(df_libros.head(12))


Extracción finalizada. Total de libros: 12


,Código,Nombre,Valor,URL
0,614112,El extranjero,"$52,000.00",https://www.panamericana.com.co/el-extranjero-...
1,606084,Harry Potter y la piedra filosofal,"$52,000.00",https://www.panamericana.com.co/harry-potter-y...
2,716793,Un amor bonito,"$59,000.00",https://www.panamericana.com.co/un-amor-bonito...
3,727684,La biblia de los caídos: tomo 3 del testamento...,"$58,000.00",https://www.panamericana.com.co/la-biblia-de-l...
4,598468,El príncipe,"$45,000.00",https://www.panamericana.com.co/el-principe-59...
5,660740,La paciente silenciosa,"$72,000.00",https://www.panamericana.com.co/la-paciente-si...
6,661275,Cómo hacer que te pasen cosas buenas,"$65,000.00",https://www.panamericana.com.co/como-hacer-que...
7,459221,Cien años de soledad,"$49,000.00",https://www.panamericana.com.co/cien-anos-de-s...
8,535690,Este dolor no es mío,"$82,000.00",https://www.panamericana.com.co/este-dolor-no-...
9,720796,Villanos 9: la más bella de todas,"$45,000.00",https://www.panamericana.com.co/villanos-9-la-...


In [ ]:
nombre_archivo = "libros_panamericana_resumen.csv"

df_libros.to_csv(
    nombre_archivo,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

files.download(nombre_archivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>